# 06 — High-Cost Classification

**Objective** — identify high-cost hospitalisation events using a threshold learned from training charges only.

**Input** — cleaned analytical dataset and leakage-safe preprocessing.

**Output** — selected classifier, train-derived threshold, and final test metrics.

## 1. Prepare the Grouped Split and Risk Labels

In [ ]:
import pandas as pd
from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_classification_candidates, compare_models_cv
from src.evaluation import classification_metrics
df = pd.read_csv('data/processed/analytical_dataset.csv')
X, y, groups = prepare_feature_frame(df)
Xtr, Xte, ytr, yte, gtr, gte = grouped_train_test_split(X, y, groups)
ytr_c, yte_c, threshold = derive_high_cost_labels(ytr, yte)
preprocessor = build_preprocessor(Xtr)

## 2. Compare Candidate Models

In [ ]:
models = get_classification_candidates(preprocessor)
comparison = compare_models_cv(models, Xtr, ytr_c, gtr, 'classification')
comparison

## 3. Evaluate the Selected Model

In [ ]:
best_name = comparison.iloc[0]['model']
best_model = models[best_name].fit(Xtr, ytr_c)
score = best_model.predict_proba(Xte)[:, 1]
pred = (score >= 0.5).astype(int)
{'threshold': threshold, 'selected_model': best_name, **classification_metrics(yte_c, score, pred)}

## Takeaways

PR-AUC is the primary selection metric because the positive high-cost class is intentionally rare. Recall is also emphasized because false negatives represent high-cost cases that the classifier fails to flag.